In [ ]:
# ============================================================
# CARGA DE DATOS — HOJAS 5A y 5B
# Se leen ambas hojas del mismo documento de Google Sheets
# y se almacenan en DataFrames separados para su análisis.
# ============================================================

# Importación de librerías
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Autenticación en Google
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Apertura del documento de Google Sheets
file_id = '11vJXajjCLX1-LJeQ3FXsCo_yr-zZv49mnHq-6MDlhqw'
sh = gc.open_by_key(file_id)

# ---- Lectura de la hoja 5A ----
datos_5A = sh.worksheet('5A').get('A1:E23')
df_5A = pd.DataFrame(datos_5A[1:], columns=datos_5A[0])

# ---- Lectura de la hoja 5B ----
datos_5B = sh.worksheet('5B').get('A1:E23')
df_5B = pd.DataFrame(datos_5B[1:], columns=datos_5B[0])

# Verificación de los datos cargados
print("=== Grupo 5A ===")
print(df_5A.head())
print(f"\nTotal de alumnos en 5A: {len(df_5A)}")

print("\n=== Grupo 5B ===")
print(df_5B.head())
print(f"\nTotal de alumnos en 5B: {len(df_5B)}")

In [ ]:
# ============================================================
# FUNCIÓN AUXILIAR: calcular_regresion(df, nombre_grupo)
# Encapsula todos los cálculos de regresión lineal para un grupo.
# Recibe un DataFrame y devuelve un diccionario con todos
# los resultados necesarios para graficar y comparar.
# ============================================================

def calcular_regresion(df, nombre_grupo):
    """
    Calcula la regresión lineal simple para un grupo dado.

    Parámetros:
        df           : DataFrame con columnas 'Altura (cm)' y 'Peso (KG)'
        nombre_grupo : Etiqueta del grupo (ej. '5A', '5B')

    Retorna:
        dict con: x, y, n, m, b, r, Se, media_x, media_y, std_x, std_y
    """

    # Extracción y conversión numérica de las columnas
    x = pd.to_numeric(df['Altura (cm)'], errors='coerce')  # Variable independiente
    y = pd.to_numeric(df['Peso (KG)'],   errors='coerce')  # Variable dependiente

    # Número de datos
    n = len(x)

    # Sumas necesarias para la fórmula de Pearson
    sum_x  = x.sum()
    sum_y  = y.sum()
    sum_xy = (x * y).sum()
    sum_x2 = (x ** 2).sum()
    sum_y2 = (y ** 2).sum()

    # Coeficiente de correlación de Pearson
    r = (n * sum_xy - sum_x * sum_y) / \
        np.sqrt((n * sum_x2 - sum_x**2) * (n * sum_y2 - sum_y**2))

    # Pendiente e intercepto de la recta de regresión
    m = (n * sum_xy - sum_x * sum_y) / (n * sum_x2 - sum_x**2)
    b = (sum_y - m * sum_x) / n

    # Error estándar de estimación
    y_pred = m * x + b
    Se = np.sqrt(((y - y_pred)**2).sum() / (n - 2))

    # Estadísticas descriptivas
    media_x = x.mean()
    media_y = y.mean()
    std_x   = x.std()
    std_y   = y.std()

    # Impresión del resumen del grupo
    print(f"\n{'='*40}")
    print(f"  RESULTADOS — Grupo {nombre_grupo}")
    print(f"{'='*40}")
    print(f"  n (alumnos)        = {n}")
    print(f"  Media X (Altura)   = {media_x:.4f} cm")
    print(f"  Media Y (Peso)     = {media_y:.4f} kg")
    print(f"  Desv. Est. X       = {std_x:.4f}")
    print(f"  Desv. Est. Y       = {std_y:.4f}")
    print(f"  Pendiente    m     = {m:.4f}")
    print(f"  Intercepto   b     = {b:.4f}")
    print(f"  Ecuación           = y = {m:.4f}x + ({b:.4f})")
    print(f"  Coef. correlación r= {r:.4f}")
    print(f"  Error estándar Se  = {Se:.4f} kg")

    return {
        'nombre': nombre_grupo,
        'x': x, 'y': y,
        'n': n, 'm': m, 'b': b, 'r': r, 'Se': Se,
        'media_x': media_x, 'media_y': media_y,
        'std_x': std_x,     'std_y': std_y
    }


# ---- Aplicar la función a cada grupo ----
res_5A = calcular_regresion(df_5A, '5A')
res_5B = calcular_regresion(df_5B, '5B')

In [ ]:
# ============================================================
# GRÁFICA 1: NUBES DE PUNTOS INDIVIDUALES (lado a lado)
# Se muestran los diagramas de dispersión de cada grupo
# en subplots separados para comparar su distribución.
# ============================================================

# Crear figura con 2 subplots horizontales (1 fila, 2 columnas)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Configuración de cada grupo: (resultado, color, eje)
configs = [
    (res_5A, 'steelblue', axes[0]),
    (res_5B, 'tomato',    axes[1])
]

for res, color, ax in configs:
    # Diagrama de dispersión del grupo
    ax.scatter(res['x'], res['y'],
               color=color, edgecolors='black', s=60)

    # Etiquetas y título de cada subplot
    ax.set_xlabel('Altura (cm)')
    ax.set_ylabel('Peso (kg)')
    ax.set_title(f"Nube de Puntos — Grupo {res['nombre']}")
    ax.grid(True)

# Ajuste de espaciado entre subplots
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRÁFICA 2: REGRESIÓN INDIVIDUAL CON MÁRGENES DE ERROR
# Para cada grupo se muestra la recta de regresión y las
# bandas ±Se en subplots separados.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs = [
    (res_5A, 'steelblue', axes[0]),
    (res_5B, 'tomato',    axes[1])
]

for res, color, ax in configs:
    m, b, Se = res['m'], res['b'], res['Se']

    # Puntos para dibujar la recta (rango del grupo)
    x_linea = np.linspace(res['x'].min(), res['x'].max(), 100)
    y_linea = m * x_linea + b

    # Datos reales
    ax.scatter(res['x'], res['y'],
               color='dimgray', edgecolors='black', s=60,
               label='Datos reales', zorder=5)

    # Recta de regresión
    ax.plot(x_linea, y_linea,
            color=color, linewidth=2, linestyle='-',
            label=f'y = {m:.2f}x + {b:.2f}')

    # Bandas de error ±Se
    ax.plot(x_linea, y_linea + Se,
            color='gray', linewidth=1.5, linestyle='--',
            label=f'+{Se:.1f} kg')
    ax.plot(x_linea, y_linea - Se,
            color='gray', linewidth=1.5, linestyle='--',
            label=f'-{Se:.1f} kg')

    ax.set_xlabel('Altura (cm)')
    ax.set_ylabel('Peso (kg)')
    ax.set_title(f"Regresión Lineal con Error — Grupo {res['nombre']}")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRÁFICA 3: COMPARATIVA — AMBOS GRUPOS EN UNA SOLA GRÁFICA
# Se superponen los datos y rectas de regresión de 5A y 5B
# para facilitar la comparación visual directa.
# ============================================================

plt.figure(figsize=(10, 6))

configs = [
    (res_5A, 'steelblue', 'o'),   # Grupo 5A: azul, círculos
    (res_5B, 'tomato',    's')    # Grupo 5B: rojo, cuadrados
]

# Rango global de X para que las rectas se extiendan por igual
x_min_global = min(res_5A['x'].min(), res_5B['x'].min())
x_max_global = max(res_5A['x'].max(), res_5B['x'].max())
x_linea = np.linspace(x_min_global, x_max_global, 100)

for res, color, marker in configs:
    m, b = res['m'], res['b']
    nombre = res['nombre']

    # Datos reales del grupo
    plt.scatter(res['x'], res['y'],
                color=color, edgecolors='black',
                s=60, marker=marker,
                label=f"Datos {nombre}", zorder=5, alpha=0.8)

    # Recta de regresión del grupo
    plt.plot(x_linea, m * x_linea + b,
             color=color, linewidth=2,
             label=f"Regresión {nombre}: y={m:.2f}x+{b:.2f}  (r={res['r']:.4f})")

plt.xlabel('Altura (cm)')
plt.ylabel('Peso (kg)')
plt.title('Comparación de Regresión Lineal: Grupo 5A vs Grupo 5B')
plt.legend(fontsize=9)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TABLA COMPARATIVA DE RESULTADOS
# Resume en un DataFrame los indicadores clave de cada grupo
# para facilitar la comparación numérica.
# ============================================================

tabla = pd.DataFrame({
    'Grupo':              [res_5A['nombre'],   res_5B['nombre']],
    'n (alumnos)':        [res_5A['n'],         res_5B['n']],
    'Media Altura (cm)':  [round(res_5A['media_x'], 4), round(res_5B['media_x'], 4)],
    'Media Peso (kg)':    [round(res_5A['media_y'], 4), round(res_5B['media_y'], 4)],
    'Pendiente m':        [round(res_5A['m'], 4),       round(res_5B['m'], 4)],
    'Intercepto b':       [round(res_5A['b'], 4),       round(res_5B['b'], 4)],
    'Correlación r':      [round(res_5A['r'], 4),       round(res_5B['r'], 4)],
    'Error estándar Se':  [round(res_5A['Se'], 4),      round(res_5B['Se'], 4)],
})

# Mostrar la tabla transpuesta para mejor legibilidad
print(tabla.set_index('Grupo').T.to_string())

# ---- Interpretación automática ----
print("\n" + "="*50)
print("INTERPRETACIÓN")
print("="*50)

# ¿Cuál grupo tiene mayor correlación?
mejor_r = '5A' if abs(res_5A['r']) >= abs(res_5B['r']) else '5B'
print(f"• Mayor correlación lineal : Grupo {mejor_r}")

# ¿Cuál grupo tiene menor error de estimación?
mejor_Se = '5A' if res_5A['Se'] <= res_5B['Se'] else '5B'
print(f"• Menor error estándar (Se): Grupo {mejor_Se}")

# Diferencia entre pendientes
diff_m = abs(res_5A['m'] - res_5B['m'])
print(f"• Diferencia entre pendientes: {diff_m:.4f}")
print("  (Cuanto más cercana a 0, más similares son las relaciones Altura-Peso)")